## NBA 3.1 | LAB Patient Features

Builds First Occurrence, Recency, and Frequency features for Lab claims.
No Hospitalization features for Lab (DX/PX only).

**Input file:** `lab` (`lab_input_claims_12M_and_first_occ_features_input`, 389M rows, 2021-07-01→2026-08-31)

**Output:** `.../kerendia_nba_3_1_volume/notebook_output/patient_features/lab/`

**Skill:** `databricks-skills/patient_features_skills/SKILL.md` (Sections 3–5 incl. NBA 3.1 removal notes). Builders come from `00_Patient_Feature_Helpers_31` (optimized, validated). Section 13 add-on features are built in `01_Additional_Patient_Features`.

**NBA 3.1 changes applied:** censored first-occurrence events dropped / capped at 365 d + `_FIRST_CLAIM_BEFORE_WINDOW` flag; unsuffixed recency totals never written; `LOGRATIO_FREQ_*` (N = 3, 6); unique patients for 90/180/360 d only; `FREQ_*_IN_LAST_*` dropped where `NUM_OF_PATIENTS_*` exists. Lab has no CLAIM_ID, so counts use rows. Frequency output = one row per HCP x cohort at `SVC_DT_M = LOOKBACK_END_DATE`.

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"

In [0]:
%run "/Workspace/Users/sehar.aejaz.ext@bayer.com/kerendia_nba_3_1_develop/NBA 3.1 - Dev/02_Input_Data_Creation/00_event_list_config"

In [0]:
%run "/Workspace/Users/sehar.aejaz.ext@bayer.com/kerendia_nba_3_1_develop/NBA 3.1 - Dev/00_Helper_Notebooks/00_cohort_generation"

In [0]:
%run "/Workspace/Users/sehar.aejaz.ext@bayer.com/kerendia_nba_3_1_develop/NBA 3.1 - Dev/03_patient_features/Final Notebooks/00_Patient_Feature_Helpers_31"

In [0]:
import re
from functools import reduce
from pyspark.sql import functions as F, Window

FEATURE_FAMILY = "claims"
CLAIM_TYPE     = "lab"
EVENT_TYPE_VAL = "lab"

# Lab: single file for all features (full history starting 2021-07-01)
LAB_PATH = INPUT_PATHS_31["lab"]

OUTPUT_BASE  = ("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/"
                "kerendia_nba_3/kerendia_nba_3_1_volume/notebook_output/patient_features/lab")
FO_OUT   = f"{OUTPUT_BASE}/lab_first_occurrence_features"
REC_OUT  = f"{OUTPUT_BASE}/lab_recency_features"
FREQ_OUT = f"{OUTPUT_BASE}/lab_frequency_features"

dbutils.widgets.text("TEST_HCP_SAMPLE", "0", "HCP sample size (0=full)")
TEST_HCP_SAMPLE = int(dbutils.widgets.get("TEST_HCP_SAMPLE"))
if TEST_HCP_SAMPLE > 0:   # sample runs write to their own folder so full-universe outputs are never overwritten
    OUTPUT_BASE = f"{OUTPUT_BASE}/_test_sample_{TEST_HCP_SAMPLE}"
    FO_OUT, REC_OUT, FREQ_OUT = [f"{OUTPUT_BASE}/{p.rsplit('/', 1)[1]}" for p in (FO_OUT, REC_OUT, FREQ_OUT)]
RECENCY_WINDOWS = [30, 60, 90, 180, 270, 360]
LAG_WINDOWS     = [1, 2, 3, 6, 12]
CHANGE_WINDOWS  = [3, 6]
PATIENT_WINDOWS = [90, 180, 360]          # NBA 3.1 removal note: 90/180/360 only
DROP_FREQ_WITH_PATIENT_EQUIV = True       # NBA 3.1: drop FREQ_*_IN_LAST_* when a NUM_OF_PATIENTS_* equivalent exists
TMP_BASE        = f"{OUTPUT_BASE}/_tmp"   # Delta checkpoints for intermediates (Section 7.4), removed at the end

print(f"LAB_PATH    : {LAB_PATH}")
print(f"OUTPUT_BASE : {OUTPUT_BASE}")

In [0]:
cohorts = CohortGenerator(FEATURE_FAMILY).load()
meta = cohorts.attrs["metadata"]
assert FEATURE_FAMILY == "claims" and meta["feature_family"] == "claims" and meta["offset_days"] == 0
assert list(cohorts["cohort_id"]) == [f"c{i}" for i in range(1, 29)]
c1 = cohorts.iloc[0]
assert (str(c1["lookback_start_date"]), str(c1["lookback_end_date"])) == ("2023-06-01", "2024-05-01")
print(f"Cohorts loaded: {len(cohorts)} rows")

In [0]:
HCP_UNIVERSE_PATH = ("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/"
                     "kerendia_nba_3/kerendia_nba_3_1_volume/HCP_Universe")
hcp_universe_df = spark.read.parquet(HCP_UNIVERSE_PATH).select("PROVIDER_ID").distinct()
if TEST_HCP_SAMPLE > 0:
    hcp_universe_df = (hcp_universe_df.orderBy(F.xxhash64("PROVIDER_ID"), "PROVIDER_ID")
                                      .limit(TEST_HCP_SAMPLE))
hcp_universe = F.broadcast(hcp_universe_df)
lab_event_keys = event_groups("lab")
print(f"Universe HCPs: {'TEST:'+str(TEST_HCP_SAMPLE) if TEST_HCP_SAMPLE > 0 else '314,920 (full)'}")
print(f"Lab EVENT_GROUPs: {len(lab_event_keys)}")

In [0]:
claims_all = (spark.read.format("delta").load(LAB_PATH)
               .join(hcp_universe, "PROVIDER_ID", "left_semi"))
claims_all = standardize(claims_all, EVENT_TYPE_VAL)
validate_event_input(claims_all, CLAIM_TYPE)
# Lab data has no CLAIM_ID column — keep only the columns standardize() produces
claims_freq = (claims_all
               .withColumn("SVC_DT_M", F.trunc(F.col("EVENT_DATE"), "month"))
               .select("PROVIDER_ID", "PATIENT_ID", "EVENT_NAME", "SVC_DT_M",
                       "EVENT_DATE"))
print("lab loaded and standardized")

In [0]:
# Lab data starts 2021-07-01; the censoring rule measures share of HCPs first seen within 60 d of that start.
fo, fo_censor_report = build_first_occurrence(claims_all, lab_event_keys, cohorts, tmp_path=f"{TMP_BASE}/_fo_pre")
print(fo_censor_report["RULE"].value_counts().to_string())
print("Dropped (left-censored):", fo_censor_report.loc[fo_censor_report["RULE"] == "DROP_CENSORED", "EVENT_SAN"].tolist())
(spark.createDataFrame(fo_censor_report.astype({"DATA_START": str}))
      .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
      .save(f"{OUTPUT_BASE}/lab_first_occurrence_censor_report"))
(fo.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
   .partitionBy("COHORT").save(FO_OUT))
print(f"Written: {FO_OUT}")

In [0]:
# Lab has no CLAIM_ID -> count rows (count_claim_ids=False); suffix LAB_DATA
recency = build_recency(claims_all, lab_event_keys, cohorts, suffix="LAB_DATA", windows=RECENCY_WINDOWS,
                        count_claim_ids=False, tmp_base=f"{TMP_BASE}/rec")
(recency.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
        .partitionBy("COHORT").save(REC_OUT))
print(f"Written: {REC_OUT}")

In [0]:
frequency = build_frequency(claims_freq, lab_event_keys, cohorts, patient_prefix="DID_LAB_TEST",
                            lag_windows=LAG_WINDOWS, change_windows=CHANGE_WINDOWS,
                            patient_windows=PATIENT_WINDOWS,
                            drop_freq_with_patient_equiv=DROP_FREQ_WITH_PATIENT_EQUIV,
                            tmp_base=f"{TMP_BASE}/freq")
(frequency.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
          .partitionBy("COHORT").save(FREQ_OUT))
print(f"Written: {FREQ_OUT} | columns: {len(frequency.columns)}")

In [0]:
for path, label in [(FO_OUT, "First Occurrence"), (REC_OUT, "Recency"), (FREQ_OUT, "Frequency")]:
    df = spark.read.format("delta").load(path)
    cohort_ids = {r[0] for r in df.select("COHORT").distinct().collect()}
    expected   = set(cohorts["cohort_id"])
    n_outside  = (df.select("PROVIDER_ID").distinct()
                    .join(hcp_universe_df, "PROVIDER_ID", "left_anti").count())
    print(f"{label}: {df.count():,} rows | cohorts={len(cohort_ids)}/28 "
          f"| missing={expected-cohort_ids or 'none'} | outside universe={n_outside}")
dbutils.fs.rm(TMP_BASE, True)   # drop intermediate checkpoints
dbutils.notebook.exit("LAB patient features written: First Occurrence, Recency, Frequency")